# Assistant vocal de FAQ : démonstration

Posez une question **à l'oral** sur les transferts d'argent Western Union au Sénégal :
l'assistant la transcrit (Whisper), retrouve la question la plus proche dans une FAQ
de 30 entrées (embeddings e5), puis lit la réponse à voix haute (Edge TTS).
S'il n'est pas assez sûr, il répond « je ne sais pas » plutôt que d'inventer.

**Avant de commencer :** Exécution > Modifier le type d'exécution > **GPU T4**.
Sur CPU, tout fonctionne, mais la transcription est environ 20 fois plus lente.

> Projet pédagogique, non affilié à Western Union.

## 1. Installation

In [ ]:
import os

DEPOT = "https://github.com/khadijagueye235/Assistant-vocal-de-FAQ"
DOSSIER = "/content/Assistant-vocal-de-FAQ"
if os.path.isdir(DOSSIER):
    !git -C {DOSSIER} pull -q
else:
    !git clone -q {DEPOT} {DOSSIER}
%cd {DOSSIER}
!pip install -q -r requirements.txt
!nvidia-smi -L || echo "Pas de GPU : l'ASR sera lente (voir l'encadré en haut)."

## 2. Vérification : tests automatiques

In [ ]:
!python -m pytest -q

## 3. Démarrage

Charge Whisper et le modèle d'embeddings, puis synthétise une fois les 32 messages
possibles (30 réponses, « je ne sais pas », « rien entendu ») : ensuite, la voix
ne coûte plus que quelques millisecondes par question.

In [ ]:
import time

from IPython.display import Audio, display

from src.pipeline import prechauffer, repondre

debut = time.perf_counter()
prechauffer()
print(f"Prêt en {time.perf_counter() - debut:.1f} s")


def afficher(r):
    lat = r.latences
    print(f"Transcription : « {r.transcription} »")
    if r.a_trouve:
        print(f"Question retrouvée : n°{r.id_question} « {r.question_trouvee} »")
        print(f"Score : {r.score:.3f}")
    else:
        score = f"{r.score:.3f}" if r.score is not None else "aucun"
        print(f"Aucune question assez proche (meilleur score : {score})")
    print(f"Réponse : {r.reponse_texte}")
    print(
        f"Latences : ASR {lat.asr_s:.2f} s · recherche {lat.recherche_s * 1000:.0f} ms · "
        f"TTS {lat.tts_s * 1000:.0f} ms · total {lat.total_s:.2f} s"
    )
    if r.audio_reponse:
        display(Audio(str(r.audio_reponse)))

## 4. Exemples (voix synthétiques)

In [ ]:
from src.tts import parler

questions = [
    "Euh, c'est quoi le MTCN ?",
    "Ça coûte combien pour recevoir de l'argent ?",
    "Qui a gagné le match du Sénégal hier ?",  # hors FAQ : l'assistant doit refuser
]
for i, q in enumerate(questions, 1):
    # Une autre voix (Henri) pose la question ; Denise répond.
    question_audio = parler(q, voix="fr-FR-HenriNeural", cache=False)
    print()
    print(f"=== Question {i} ===")
    display(Audio(str(question_audio)))
    afficher(repondre(question_audio))

## 5. Votre propre question

Enregistrez-vous (téléphone, enregistreur de Windows...) et téléversez le fichier.
Il reste dans `/content`, effacé à la fin de la session ; il n'est jamais envoyé
sur GitHub.

In [ ]:
from google.colab import files

for nom, contenu in files.upload().items():
    chemin = f"/content/{nom}"
    with open(chemin, "wb") as fichier:
        fichier.write(contenu)
    os.remove(nom)  # files.upload garde aussi une copie dans le dossier du dépôt
    afficher(repondre(chemin))

## 6. (Optionnel) Évaluation sur le jeu de développement audio

105 questions lues par 8 voix synthétiques : taux d'erreur de l'ASR, précision
de la recherche, bilan du seuil « je ne sais pas ». Environ 3 minutes sur GPU.

In [ ]:
!python -m scripts.generer_dev_audio
!python -m scripts.evaluer_dev_audio